# Editing a sequence graph

`SequenceGraph` has these editing methods:

| Method | Where | Result |
|--------|-------|--------|
| `insert(sequence, after=position)` | a position | adds `sequence` right after it |
| `insert(sequence, before=position)` | a position | adds `sequence` right before it |
| `insert(sequence, after=left, before=right)` | two neighbouring positions | adds `sequence` where `left` reads into `right` |
| `replace(target, sequence)` | a stretch of sequence | swaps that stretch for `sequence` |
| `delete(target)` | a stretch of sequence | removes that stretch |

A `Position` is a point in a node's sequence: a node, an offset, and a strand. `locus.start()` and
`locus.end()` give the first and last position of a locus, on the locus's strand, so "after" follows
the locus's reading direction.

Several positions held together form a `SuperPosition`, which `insert()` also accepts.
Use one to move along the sequence, or to insert at several places at once. Build it from positions,
attach it to a graph, and add or subtract steps: `SuperPosition(locus.start()).on(sg) + 3` is the
fourth position of the locus. Where the graph branches, a step can reach one position on each
branch, and an insertion given only `after` or only `before` lands on all of them. Give both to pick
out one junction: `left` must read directly into `right`, and the new sequence goes only there.

A target for `replace()` and `delete()` can be a region string such as `"pUC19:100-110"`, a
`Locus` returned by `search()`, or an `Annotation`.

Every call is recorded as its own operation in the repository history. If an edit is refused,
nothing changes and nothing is recorded.

This notebook works through a few edits to the pUC19 cloning vector:

1. Clone a FLAG tag into the EcoRI site of the multiple cloning site.
2. Change the start codon of *bla* (β-lactamase) from ATG to GTG.
3. Remove the CAP binding site upstream of *lacZα*.
4. Keep two alternative restriction sites side by side with `stack=True`.

## Setup

In [ ]:
import pathlib
import tempfile

import gen

EXAMPLES_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".").resolve()

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-editing-"))
repo = gen.Repository(str(WORK_DIR))
reference = repo.import_genbank(str(EXAMPLES_DIR / "puc19.gbk"), sample="reference")
sg_reference = reference[0]

`export_fasta()` writes the sequence as it currently reads end to end. This helper returns that
sequence as a string so we can look at it before and after each edit.

In [ ]:
def current_sequence(sg):
    path = WORK_DIR / f"{sg.sample_name}.fa"
    sg.export_fasta(str(path))
    lines = path.read_text().splitlines()
    return "".join(line for line in lines if not line.startswith(">")).upper()


reference_sequence = current_sequence(sg_reference)
len(reference_sequence)

## Edit a copy of the sample

Edits change a sequence graph in place. To keep the original, copy the sample first and edit the
copy. `copy()` needs a sample name that doesn't exist yet.

In [ ]:
edited = reference.copy("edited")
sg = edited[0]

## Insert at a search hit

EcoRI cuts at `GAATTC`. That site reads the same on both strands, so `search()` returns one hit per
strand; we take the forward one. The tag goes right after the last position of the site.

In [ ]:
[eco_ri] = [hit for hit in sg.search("GAATTC") if hit.strand == "+"]

flag_tag = sg.insert("GACTACAAAGACGATGACGACAAG", after=eco_ri.end())

print("before:", reference_sequence[390:440])
print("after: ", current_sequence(sg)[390:440])

`insert()` and `replace()` return the `Locus` of the sequence they added. That locus is a
convenient target for the next edit: here a stop codon goes straight after the tag. The optional
`message` sets the text recorded in the history.

In [ ]:
sg.insert("TAA", after=flag_tag.end(), message="Add stop codon after FLAG tag")

print("after:", current_sequence(sg)[390:440])

## Replace part of an annotation

*bla* sits on the reverse strand. `Locus.slice()` counts in reading order along the locus's own
strand, so `slice(0, 3)` is the start codon however the gene is oriented. The replacement sequence
is read on that strand too.

The annotation was imported before we inserted the tag, and its locus still points at the same
sequence. Loci keep working after edits elsewhere in the graph.

In [ ]:
bla = next(
    annotation
    for annotation in sg.annotations
    if annotation.name == "bla" and len(annotation.locus) == 861
)
start_codon = bla.locus.slice(0, 3)

sg.replace(start_codon, "GTG")

# The exported sequence is the forward strand, where the start codon reads as its reverse
# complement: CAT becomes CAC. The tag inserted upstream shifts everything by 27 positions.
shift = len(flag_tag) + 3
print("reference:", reference_sequence[2476:2496])
print("edited:   ", current_sequence(sg)[2476 + shift : 2496 + shift])

## Delete an annotation

An `Annotation` can be passed straight to an editing method. `delete()` returns nothing, since
there is no new sequence to point at.

In [ ]:
cap_site = next(
    annotation for annotation in sg.annotations if annotation.name == "CAP protein binding site"
)

sg.delete(cap_site)

edited_sequence = current_sequence(sg)
print("reference:", reference_sequence[552:585])
print("edited:   ", edited_sequence[552 + shift : 572 + shift])
print("length:", len(reference_sequence), "->", len(edited_sequence))

## Region strings follow the current sequence

A region string is read against the sequence as it is now, not as it was when the sample was
imported. After 27 positions were inserted at the EcoRI site and 13 removed at the CAP site,
`"pUC19:1900-1910"` names a different stretch than it did in the reference. To keep pointing at the
same sequence across edits, hold on to a `Locus` or an `Annotation` instead.

In [ ]:
net_shift = len(edited_sequence) - len(reference_sequence)
print("pUC19:1900-1910 in reference:", reference_sequence[1900:1910])
print("pUC19:1900-1910 in edited:   ", edited_sequence[1900:1910])
print(f"the same stretch in edited:   {edited_sequence[1900 + net_shift : 1910 + net_shift]} (shifted by {net_shift})")

## Refused edits

An edit that can't be applied raises `ValueError` and leaves both the graph and the history as
they were. Two common cases: targeting sequence that an earlier edit already removed, and an
`insert()` given `after` and `before` positions that are not neighbours. `insert()` only adds
sequence, so it refuses to span the FLAG tag; `replace()` is the call that swaps a stretch out.

In [ ]:
operations_before = len(repo.get_operations())

try:
    sg.delete(cap_site)
except ValueError as error:
    print("delete again:", error)

try:
    sg.insert("AAA", after=flag_tag.start(), before=flag_tag.end())
except ValueError as error:
    print("positions that are not neighbours:", error)

print("operations recorded:", len(repo.get_operations()) - operations_before)

## Keep alternatives side by side with `stack=True`

A plain edit changes what the sequence reads. The original sequence drops out of it, which is why
deleting the CAP site a second time was refused above.

Every editing method accepts `stack=True`, which adds the new sequence as an alternative next to
what is already there. The graph gains a bubble, the original stays a valid target for more edits,
and the exported sequence doesn't change, since no option has been picked.

Here a fresh copy of the reference gets two alternatives at the EcoRI site: a BamHI site and a
HindIII site. The second `replace()` targets the same EcoRI locus as the first, which only works
because the first one was stacked.

In [ ]:
variants = reference.copy("variants")
sg_variants = variants[0]

[eco_ri] = [hit for hit in sg_variants.search("GAATTC") if hit.strand == "+"]
bam_hi = sg_variants.replace(eco_ri, "GGATCC", stack=True)
hind_iii = sg_variants.replace(eco_ri, "AAGCTT", stack=True)

print("exported sequence unchanged:", current_sequence(sg_variants) == reference_sequence)

In [ ]:
sg_variants

The EcoRI option is still there, so the locus found before stacking is still a valid target.
`eco_ri.start()` is the first position of that option, and only the position before the options
reads into it, so an insertion before it goes on the EcoRI route and leaves the BamHI and HindIII
options alone. The exported sequence follows the EcoRI route,
so it shows the change.

In [ ]:
sg_variants.insert("CC", before=eco_ri.start())

print("before:", reference_sequence[390:410])
print("after: ", current_sequence(sg_variants)[390:412])

## History

Each edit above is its own operation, newest first. Edits without a `message` get a generated one.

In [ ]:
for operation in repo.get_operations()[:10]:
    print(operation.message)